# Поиск объявлений об услугах: 50 кандидатов на запрос

Задача — найти в корпусе объявления, которые пользователь выбрал по короткому запросу.
Если подходящее объявление не попало в 50 кандидатов, следующий этап поиска уже не сможет его вернуть.
Поэтому основная метрика здесь — **Recall@50**, усреднённая по запросам.

В этой папке находится решение, для которого платформа показала **0.882835**.
Отправленный файл — `answer.csv`, его контрольная сумма проверяется ниже.
Ноутбук объясняет решение и служит точкой входа в запуск. Полный код вынесен в обычные Python-модули:
так индексы, обучение и предсказание можно воспроизвести без ручного выполнения отдельных ячеек.

## Как запустить

1. Установить зависимости: `python -m pip install -r requirements-notebook.txt`.
2. Распаковать три Parquet из задания в `data/`.
3. Проверить веса: `python setup_assets.py --verify-only`. В полном архиве решения они уже есть.
   Если веса не были включены при копировании проекта, `python setup_assets.py` загрузит зафиксированную версию.
4. Выполнить **Run All**.

По умолчанию выполняются анализ данных и проверки готового результата. Повторный расчёт всех
кандидатов включается флагом `RUN_REPRODUCE`. Он запускает настоящий `reproduce.py`,
а не копирует приложенный ответ. После подготовки зависимостей предсказание работает офлайн.

Если данные лежат отдельно, можно изменить `DATA_DIR` или задать переменную `AVITO_DATA_DIR`.
В выводе ячеек используются агрегаты; тексты описаний, контакты и координаты не выводятся.

In [1]:
from pathlib import Path
import hashlib
import json
import os
import pickle
import subprocess
import sys

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from IPython.display import display

ROOT = Path.cwd()
if not (ROOT / 'solution_pipeline.py').is_file():
    raise RuntimeError('Откройте ноутбук из папки проекта рядом с solution_pipeline.py.')

DATA_DIR = Path(os.environ.get('AVITO_DATA_DIR', str(ROOT / 'data'))).expanduser()
RUN_REPRODUCE = False
DEVICE = 'cpu'

required = ['train.parquet', 'benchmark_items.parquet', 'benchmark_queries.parquet']
missing = [name for name in required if not (DATA_DIR / name).is_file()]
if missing:
    raise FileNotFoundError('Добавьте в DATA_DIR файлы: ' + ', '.join(missing))

def read_json(relative_path):
    return json.loads((ROOT / relative_path).read_text(encoding='utf-8'))

def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as file:
        for block in iter(lambda: file.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

EXPECTED_ANSWER_SHA = 'ef4ba4b73e9cbb7821d4b7c2a1899b97f8c962adde06a81a70f81dd51c9910ea'
config = read_json('models/selected_config.json')
metrics = read_json('results/metrics.json')
holdout = read_json('results/fresh_holdout_metrics.json')
platform = read_json('results/platform_result.json')
training = read_json('models/training_report.json')
feature_names = read_json('models/feature_names.json')
print('Три исходные таблицы найдены. Режим:', 'полный расчёт' if RUN_REPRODUCE else 'анализ и проверки')

Три исходные таблицы найдены. Режим: анализ и проверки


## 1. Метрика и формат ответа

Для запроса с двумя известными положительными объявлениями попадание одного даёт Recall = 0.5.
Сначала считается полнота каждого запроса, затем среднее. Запрос с большим числом положительных
объявлений не должен получать больший вес только из-за их количества.

В `answer.csv` ровно две колонки: `query_id` и `answer`.
Во второй колонке — идентификаторы объявлений через один пробел. Порядок внутри top-50
не влияет на Recall. Возвращаются строки идентификаторов, а не номера строк таблицы.

In [2]:
# Небольшой пример определения метрики; это не измерение качества решения.
hits = np.array([1, 1, 0])
relevant = np.array([1, 2, 1])
display(pd.DataFrame({'найдено': hits, 'известно положительных': relevant,
                      'Recall запроса': hits / relevant}))
print('Macro Recall@50:', float(np.mean(hits / relevant)))

,найдено,известно положительных,Recall запроса
0,1,1,1.0
1,1,2,0.5
2,0,1,0.0


Macro Recall@50: 0.5


## 2. Что есть в данных

`train.parquet` содержит пары «запрос — выбранное объявление». Отрицательных ответов в нём нет.
`benchmark_queries.parquet` содержит запросы, для которых нужен ответ.
`benchmark_items.parquet` — доступный для них корпус. В локальной проверке корпус строится
из всех уникальных объявлений train; обучение статистик при этом ограничивается отдельной частью взаимодействий.

In [3]:
tables = []
for name in required:
    parquet = pq.ParquetFile(DATA_DIR / name)
    tables.append({'файл': name, 'строк': parquet.metadata.num_rows,
                   'столбцов': len(parquet.schema_arrow.names),
                   'размер файла, МБ': round((DATA_DIR / name).stat().st_size / 1024**2, 2)})
display(pd.DataFrame(tables))

,файл,строк,столбцов,"размер файла, МБ"
0,train.parquet,497673,19,467.53
1,benchmark_items.parquet,189212,14,186.61
2,benchmark_queries.parquet,2452,6,0.12


In [4]:
from retrieval import SEARCH_COLS, normalize

# Для анализа взаимодействий не нужны длинные описания из train.
train = pd.read_parquet(DATA_DIR / 'train.parquet',
                        columns=SEARCH_COLS + ['item_id', 'item_location_id'])
queries = pd.read_parquet(DATA_DIR / 'benchmark_queries.parquet')
items = pd.read_parquet(DATA_DIR / 'benchmark_items.parquet')

pair_columns = SEARCH_COLS + ['item_id']
pairs = train.drop_duplicates(pair_columns)
train_texts = train.search_query.map(normalize)
benchmark_texts = queries.search_query.map(normalize)
train_ids = set(train.item_id)
quality = pd.DataFrame([
    {'показатель': 'Уникальные item_id в train', 'значение': train.item_id.nunique()},
    {'показатель': 'Повторные пары по полному набору признаков запроса и item_id',
     'значение': int(train.duplicated(pair_columns).sum())},
    {'показатель': 'Уникальные нормализованные тексты запросов train',
     'значение': train_texts.nunique()},
    {'показатель': 'Уникальные query_id benchmark', 'значение': queries.query_id.nunique()},
    {'показатель': 'Уникальные item_id benchmark', 'значение': items.item_id.nunique()},
    {'показатель': 'Объявления benchmark, встречающиеся в train',
     'значение': int(items.item_id.isin(train_ids).sum())},
    {'показатель': 'Строки benchmark с текстом запроса, известным из train',
     'значение': int(benchmark_texts.isin(set(train_texts)).sum())},
])
display(quality)
print('Доля совпадающих локаций в уникальных положительных парах:',
      round(float(pairs.search_location_id.eq(pairs.item_location_id).mean()), 4))

assert queries.query_id.is_unique and queries.query_id.str.len().eq(16).all()
assert items.item_id.is_unique and items.item_id.str.fullmatch(r'[0-9a-f]{16}').all()
print('Уникальность и строковый формат идентификаторов проверены.')

,показатель,значение
0,Уникальные item_id в train,344825
1,Повторные пары по полному набору признаков зап...,30619
2,Уникальные нормализованные тексты запросов train,73706
3,Уникальные query_id benchmark,2452
4,Уникальные item_id benchmark,189212
5,"Объявления benchmark, встречающиеся в train",18142
6,"Строки benchmark с текстом запроса, известным ...",919


Доля совпадающих локаций в уникальных положительных парах: 0.822
Уникальность и строковый формат идентификаторов проверены.


In [5]:
# Пустая строка отличается от отсутствующего значения; считаем оба случая.
missing_rows = []
for name in items.columns:
    column = items[name]
    blank = int(column.fillna('').astype(str).str.strip().eq('').sum()) if pd.api.types.is_object_dtype(column) else 0
    missing_rows.append({'поле': name, 'пропуски': int(column.isna().sum()),
                         'пустые строки, включая пропуски': blank if pd.api.types.is_object_dtype(column) else None})
display(pd.DataFrame(missing_rows))

lengths = pd.DataFrame({
    'заголовок, символы': items.item_title_raw.fillna('').str.len(),
    'описание, символы': items.item_description_raw.fillna('').str.len(),
})
display(lengths.quantile([0, .5, .9, .99, 1]).rename_axis('квантиль'))

,поле,пропуски,"пустые строки, включая пропуски"
0,item_title_raw,0,0.0
1,item_rating_reviews_count,11615,NaN
2,item_rating,17631,NaN
3,item_price,0,0.0
4,item_microcat_id,0,NaN
5,item_longitude,1,1.0
6,item_location_id,0,NaN
7,item_latitude,1,1.0
8,item_is_phone_hidden,0,NaN
9,item_is_message_forbidden,0,NaN


,"заголовок, символы","описание, символы"
квантиль,,
0.00,1.0,0.0
0.50,35.0,1054.0
0.90,48.0,3127.0
0.99,53.0,5541.0
1.00,100.0,8494.0


In [6]:
# Сравниваем уникальные тексты, чтобы частые запросы train не доминировали в сводке.
length_rows = []
for name, text in [('train', train_texts), ('benchmark', benchmark_texts)]:
    words = text.drop_duplicates().str.split().map(len)
    length_rows.append({'набор': name, 'уникальных текстов': len(words),
                         'медиана слов': float(words.median()),
                         '90% запросов, не более слов': float(words.quantile(.9)),
                         'доля запросов из одного слова': round(float(words.eq(1).mean()), 4)})
display(pd.DataFrame(length_rows))

,набор,уникальных текстов,медиана слов,"90% запросов, не более слов",доля запросов из одного слова
0,train,73706,3.0,5.0,0.0447
1,benchmark,2451,3.0,5.0,0.0437


Из этих проверок следуют несколько решений.

- Повторы одной пары не считаются независимыми наблюдениями при построении истории.
- Большая часть объявлений benchmark не встречается в train. Нельзя строить решение на запоминании item_id.
- Запросы короткие, а описания бывают длинными. Поэтому заголовок и описание обрабатываются раздельно.
- Локации поиска и выбранного объявления не всегда совпадают. Жёсткий фильтр по location_id потерял бы положительные примеры.
- Отсутствие наблюдаемого выбора не означает, что объявление не подходит. Это ограничение обучающих отрицательных примеров, а не только техническая деталь.

## 3. Как устроена локальная проверка

Разбивать отдельные строки случайно было бы слишком легко: один и тот же запрос мог бы оказаться
и в обучении, и в проверке. Здесь группы задаются **нормализованным текстом запроса**:
нижний регистр, замена «ё» на «е», нормализация пробелов и пунктуации.

Отдельная reference-часть нужна для статистик географии и классификатора микрокатегорий.
Бустинг обучается на 8 000 других текстов запросов. Для каждого такого текста выбирается один
вариант остальных признаков запроса; все известные положительные объявления этого варианта сохраняются.

Dev из 1 000 запросов используется для сравнения настроек. Ещё 1 500 новых текстов были оставлены
до фиксации модели и проверены один раз. Ранее использованные 2 000 dev/test-текстов исключены
из reference и из нового holdout; текущий dev входит в эти 2 000.

Текстовые индексы видят весь локальный корпус объявлений. Это допустимо: в поиске документы корпуса
известны заранее. При этом выборы пользователей из проверочных групп не участвуют в обучении статистик
или бустинга. Дат событий нет, поэтому это проверка на новых текстах, а не временная валидация.

In [7]:
split = metrics['split']
display(pd.DataFrame([
    {'часть': 'Reference: история локаций и intent', 'текстов запросов': split['reference_texts'],
     'назначение': 'обучение статистик; 405404 исходные строки'},
    {'часть': 'Обучение бустинга', 'текстов запросов': split['rank_train_queries'],
     'назначение': 'обучение 62 признаков пары'},
    {'часть': 'Dev', 'текстов запросов': split['dev_queries'],
     'назначение': 'выбор настроек; входит в ранее выделенные 2000'},
    {'часть': 'Отложенная проверка', 'текстов запросов': split['fresh_holdout_queries'],
     'назначение': 'одна проверка после фиксации модели'},
]))
assert (split['reference_texts'] + split['rank_train_queries'] +
        split['fresh_holdout_queries'] + split['excluded_old_dev_and_test_texts']) == train_texts.nunique()
print('Размеры групп согласованы с числом уникальных нормализованных текстов.')

,часть,текстов запросов,назначение
0,Reference: история локаций и intent,62206,обучение статистик; 405404 исходные строки
1,Обучение бустинга,8000,обучение 62 признаков пары
2,Dev,1000,выбор настроек; входит в ранее выделенные 2000
3,Отложенная проверка,1500,одна проверка после фиксации модели


Размеры групп согласованы с числом уникальных нормализованных текстов.


## 4. Поиск кандидатов

В основе — несколько простых способов найти текстовое совпадение.

1. **BM25** отдельно по заголовкам и описаниям. Он учитывает редкость слов и длину документа.
   Для русского языка используется Snowball-стемминг. В описание добавляются текстовые параметры объявления.
2. **Символьный TF-IDF** по 3–5-граммам заголовка помогает со словоформами и частью опечаток.
   Ещё один TF-IDF по словам и биграммам дополняет его.
3. **Покрытие слов запроса** показывает, сколько разных слов встретилось в заголовке или описании.
   Неизвестные слова остаются в знаменателе.
4. **E5** добавляет смысловое сходство, когда нужные слова написаны иначе.

География учитывается мягко. Используются точное совпадение location_id, расстояние до медианного
центра локации и история переходов «локация поиска → локация объявления».
Итоговый geo — максимум этих сигналов. Он не является вероятностью релевантности.
Категория также даёт мягкую поправку, а не полное исключение.

Из источников берутся кандидаты по фиксированным квотам, затем дубли удаляются.
В поставленной модели **E5 по заголовкам участвует в поиске**, а E5 с описанием добавляет признаки бустинга.

In [8]:
display(pd.DataFrame([
    {'источник': 'Основная смесь BM25 + char TF-IDF + фильтры', 'квота': 600},
    {'источник': 'Та же смесь с покрытием слов в описании', 'квота': 250},
    {'источник': 'TF-IDF слов и биграмм заголовка', 'квота': 150},
    {'источник': 'Лексический поиск с меньшим штрафом географии', 'квота': 150},
    {'источник': 'Покрытие слов в заголовке и описании', 'квота': 1000},
    {'источник': 'E5 заголовка × мягкая география и категория', 'квота': 500},
    {'источник': 'E5 заголовка + логарифмическая поправка географии', 'квота': 500},
]))
print('Квоты не суммируются в размер пула: источники часто находят одни и те же объявления.')

,источник,квота
0,Основная смесь BM25 + char TF-IDF + фильтры,600
1,Та же смесь с покрытием слов в описании,250
2,TF-IDF слов и биграмм заголовка,150
3,Лексический поиск с меньшим штрафом географии,150
4,Покрытие слов в заголовке и описании,1000
5,E5 заголовка × мягкая география и категория,500
6,E5 заголовка + логарифмическая поправка географии,500


Квоты не суммируются в размер пула: источники часто находят одни и те же объявления.


E5 — `intfloat/multilingual-e5-small`, версия зафиксирована в manifest.
У запроса префикс `query:` и предел 64 токена. Вектор запроса строится по исходному тексту,
без добавления фильтров. Для объявления используются два представления:

- заголовок с префиксом `passage:`, до 64 токенов;
- заголовок, первые 200 символов параметров и первые 500 символов описания, до 128 токенов.

Векторы нормированы, поэтому скалярное произведение равно cosine similarity.
Модель E5 не дообучается. Все эмбеддинги вычисляются локально.
Обрезка ускоряет расчёт, но может потерять узкую услугу, упомянутую только в конце описания.

## 5. Признаки бустинга

Признаки описывают пару «запрос — объявление» и положение объявления среди кандидатов.
В итоговом наборе **62 колонки**. Идентификаторы используются только как ключи.
Частота item_id в train не передаётся бустингу.

Оценки разных поисковых источников имеют разный масштаб. Поэтому рядом с исходной оценкой
передаются место в пуле и оценка относительно максимума. Эти признаки считаются **на всём пуле**,
до выборки отрицательных примеров.

Классификатор типа услуги — `MultinomialNB` на word/char TF-IDF запроса и фильтров.
Он обучается на reference; целевой класс — microcat выбранного объявления.
Его выходы используются как признаки совместимости с типом услуги, а не как готовый ответ.

In [9]:
feature_groups = {
    'Совпадение текстов': ['title', 'body', 'char', 'filters', 'word_title',
        'title_coverage', 'body_coverage', 'title_idf_coverage', 'body_idf_coverage',
        'query_in_title', 'stem_query_in_title'],
    'Оценки источников и положение в пуле': [
        source + suffix
        for source in ['baseline', 'word_title', 'coverage_body', 'coverage_both',
                       'lexical', 'geo_005', 'geo_050']
        for suffix in ['_score', '_rank', '_relative']],
    'География и надёжность её истории': ['geo', 'same_location',
        'query_center_available', 'query_prior_max', 'query_prior_locations_log1p', 'query_prior_entropy'],
    'Свойства объявления': ['title_chars', 'body_chars', 'params_chars', 'rating',
        'log_reviews', 'phone_hidden', 'message_forbidden', 'price_missing_or_negative', 'log_price'],
    'Свойства запроса и категория': ['category', 'query_chars', 'query_words',
        'has_filters', 'category_unspecified', 'query_delivery'],
    'Совместимость типа услуги': ['intent_prob', 'intent_relative', 'intent_log',
        'intent_top', 'intent_confidence'],
    'E5: cosine и сочетание с географией': ['semantic_title_cosine', 'semantic_title_geo',
        'semantic_full_cosine', 'semantic_full_geo'],
}
listed = [name for group in feature_groups.values() for name in group]
assert len(listed) == len(set(listed)) == len(feature_names) == 62
assert set(listed) == set(feature_names)
display(pd.DataFrame([{'группа': group, 'число признаков': len(names),
                       'колонки': ', '.join(names)} for group, names in feature_groups.items()]))

,группа,число признаков,колонки
0,Совпадение текстов,11,"title, body, char, filters, word_title, title_..."
1,Оценки источников и положение в пуле,21,"baseline_score, baseline_rank, baseline_relati..."
2,География и надёжность её истории,6,"geo, same_location, query_center_available, qu..."
3,Свойства объявления,9,"title_chars, body_chars, params_chars, rating,..."
4,Свойства запроса и категория,6,"category, query_chars, query_words, has_filter..."
5,Совместимость типа услуги,5,"intent_prob, intent_relative, intent_log, inte..."
6,E5: cosine и сочетание с географией,4,"semantic_title_cosine, semantic_title_geo, sem..."


## 6. Обучение

Из reference берутся статистики и intent, а бустинг учится на 8 000 отдельных запросов.
Для каждого из них строится тот же пул, который будет использоваться при предсказании.
В обучение входят 70 лучших кандидатов baseline, 120 случайных из остальных и все известные
положительные объявления **внутри пула**. Не найденные поиском positives не подмешиваются из разметки.

Если случайный отрицательный пример выбирается с вероятностью `p`, его вес равен `1/p`;
у обязательных отрицательных примеров вес 1. Для positives одного запроса суммарный вес равен 50:
каждый получает `50 / число найденных positives`. Это практическая взвешенная классификация,
**не точная оптимизация macro Recall**.

Используется `HistGradientBoostingClassifier`. Он умеет объединять числовые сигналы и обрабатывать
пропуски. Для выбора top-50 нужна хорошая относительная оценка кандидатов, а не калиброванная вероятность.
Итог: 31 лист, 500 итераций, learning rate 0.06, L2 = 10. Дополнительной поправки на baseline
или sampling score в финальном предсказании нет.

In [10]:
display(pd.DataFrame([
    {'показатель': 'Обучающих запросов', 'значение': training['rank_train_queries']},
    {'показатель': 'Строк после выборки кандидатов', 'значение': training['training_rows']},
    {'показатель': 'Известных положительных пар', 'значение': training['original_positive_pairs']},
    {'показатель': 'Положительных пар, найденных поиском', 'значение': training['retrieved_positive_pairs']},
    {'показатель': 'Положительных пар вне пула', 'значение': training['unretrieved_positive_pairs_excluded']},
]))
display(pd.DataFrame(config['hyperparameters'].items(), columns=['параметр', 'значение']))
assert training['positive_injection_outside_pool'] is False
assert config['pool'] == 'expanded'
assert config['baseline_log_weight'] == config['sampling_correction'] == 0

,показатель,значение
0,Обучающих запросов,8000
1,Строк после выборки кандидатов,1521023
2,Известных положительных пар,8468
3,"Положительных пар, найденных поиском",8244
4,Положительных пар вне пула,224


,параметр,значение
0,max_leaf_nodes,31.00
1,learning_rate,0.06
2,l2_regularization,10.00
3,max_iter,500.00
4,min_samples_leaf,40.00
5,max_bins,127.00
6,random_state,20260929.00


## 7. Проверка приложенных моделей

Перед загрузкой бустинга проверяется его SHA-256. Для E5 сверяются все файлы из manifest,
включая веса и токенизатор. Эти проверки подтверждают состав приложенных артефактов;
они сами по себе не являются повторным измерением Recall.

In [11]:
ranker_path = ROOT / 'models/ranker.pkl'
assert sha256(ranker_path) == training['ranker_sha256']
with ranker_path.open('rb') as file:
    ranker = pickle.load(file)  # Проверенный приложенный артефакт проекта.
assert ranker.n_features_in_ == len(feature_names) == 62
assert ranker.n_iter_ == config['hyperparameters']['max_iter'] == 500
assert ranker.classes_.tolist() == [0, 1]

manifest = read_json('models/model_manifest.json')
asset_ok = []
for entry in manifest['files']:
    path = ROOT / 'models/e5-small' / entry['path']
    if not path.is_file():
        raise FileNotFoundError('Веса E5 неполны: запустите python setup_assets.py.')
    asset_ok.append(path.stat().st_size == entry['bytes'] and sha256(path) == entry['sha256'])
assert all(asset_ok)
print('Бустинг: 62 признака, 500 итераций; SHA-256 совпадает.')
print('Файлов E5 проверено:', len(asset_ok), '| все размеры и SHA-256 совпадают.')
print('Версия E5:', manifest['revision'])

Бустинг: 62 признака, 500 итераций; SHA-256 совпадает.
Файлов E5 проверено: 9 | все размеры и SHA-256 совпадают.
Версия E5: 614241f622f53c4eeff9890bdc4f31cfecc418b3


## 8. Полученное качество

В таблице ниже — сохранённые результаты измерений, а не новый запуск обучения.
Dev использовался при выборе настроек, поэтому его качество оптимистичнее независимой проверки.
На отложенных 1 500 запросах была одна оценка уже зафиксированной конфигурации;
её параметры по этим ответам не подбирались. Результат платформы относится именно к приложенному CSV.

In [12]:
assert platform['answer_sha256'] == EXPECTED_ANSWER_SHA
display(pd.DataFrame([
    {'выборка': 'Dev', 'запросов': metrics['dev']['queries'],
     'лексический baseline': metrics['dev']['baseline_recall50'],
     'итоговый Recall@50': metrics['dev']['model_recall50']},
    {'выборка': 'Отложенная проверка после фиксации', 'запросов': holdout['queries'],
     'лексический baseline': holdout['baseline_macro_recall50'],
     'итоговый Recall@50': holdout['model_macro_recall50']},
    {'выборка': 'Платформа: отправленный answer.csv', 'запросов': len(queries),
     'лексический baseline': np.nan, 'итоговый Recall@50': platform['recall50']},
]).round(6))
display(pd.DataFrame([
    {'оценка на отложенной выборке': 'Recall@50 итоговой модели',
     'нижняя граница 95%': holdout['model_bootstrap_95_ci'][0],
     'верхняя граница 95%': holdout['model_bootstrap_95_ci'][1]},
    {'оценка на отложенной выборке': 'Парный прирост к baseline',
     'нижняя граница 95%': holdout['paired_improvement_bootstrap_95_ci'][0],
     'верхняя граница 95%': holdout['paired_improvement_bootstrap_95_ci'][1]},
]).round(6))
print('Отображённая оценка платформы:', platform['displayed_grade'], 'из', platform['max_grade'])

,выборка,запросов,лексический baseline,итоговый Recall@50
0,Dev,1000,0.835083,0.914333
1,Отложенная проверка после фиксации,1500,0.799667,0.894556
2,Платформа: отправленный answer.csv,2452,NaN,0.882835


,оценка на отложенной выборке,нижняя граница 95%,верхняя граница 95%
0,Recall@50 итоговой модели,0.878889,0.909003
1,Парный прирост к baseline,0.079219,0.110444


Отображённая оценка платформы: 0.88 из 1


Bootstrap-интервалы получены пересэмплированием запросов, а для прироста использовались
разности результатов на одних и тех же запросах. Они описывают неопределённость локального измерения.
Это не интервал ожидаемого результата платформы: состав запросов и корпус там отличаются.

При конечном предсказании текстовый индекс строится по benchmark-корпусу, а доступные
статистики истории и intent обучаются на полном train. Сам бустинг остаётся выбранной моделью.
Тот факт, что исходные данные для финального запуска доступны целиком, не меняет правила
локальной валидации.

## 9. Какие ошибки остались

Разбор сделан для той же итоговой модели на dev. Он разделяет две причины потери:
нужное объявление вообще не попало в пул или было найдено, но оказалось ниже 50-го места.
В кодовом выводе оставлены только агрегаты.

In [13]:
errors = read_json('results/dev_error_summary.json')
labels = {'hit': 'Найдено в top-50', 'pool_miss': 'Не попало в пул',
          'rank_miss': 'В пуле, но ниже top-50'}
display(pd.DataFrame([
    {'исход': labels[state], 'положительных объявлений': row['positives'],
     'запросов с таким исходом': row['queries'],
     'вклад в Recall / потерю': row['macro_contribution']}
    for state, row in errors['state_summary'].items()
]).round(6))
print('Потолок текущего пула на dev:', round(errors['pool_recall'], 6))
display(pd.DataFrame([
    {'места потерянных positives': interval, 'объявлений': count}
    for interval, count in errors['rank_miss_bins'].items()
]))

,исход,положительных объявлений,запросов с таким исходом,вклад в Recall / потерю
0,Найдено в top-50,966,918,0.914333
1,Не попало в пул,26,25,0.023667
2,"В пуле, но ниже top-50",64,64,0.062000


Потолок текущего пула на dev: 0.976333


,места потерянных positives,объявлений
0,51-75,16
1,76-100,7
2,101-200,24
3,201-500,12
4,501+,5


На dev 26 positives отсутствуют в пуле, ещё 64 находятся ниже top-50.
Уже найденный пул имеет потолок Recall 0.976333; основная оставшаяся потеря связана с отбором из него.

В ошибках встречаются несколько понятных ситуаций:

- **Узкая услуга в общем объявлении.** Для запроса об установке телевизора положительным может быть
  объявление мастера на час. Поэтому в решение включены описание, покрытие слов и E5,
  но общие объявления всё ещё трудно отличать друг от друга.
- **Регион поиска и место услуги не совпадают.** Это бывает у области и города, онлайн-услуг,
  трансфера в явно названном другом городе. Мягкая география снижает потери по сравнению
  с жёстким совпадением location_id, однако иногда всё равно штрафует нужный вариант.
- **Опечатки и разные написания.** Символьные признаки, стемминг и E5 помогают, но не устраняют
  все случаи транслитерации и редких названий.
- **Ограниченные тексты и разметка.** Часть услуг находится далеко в описании, а наблюдаемый выбор
  пользователя не всегда можно объяснить по заголовку. В train нет полной экспертной оценки всех
  подходящих объявлений.

Здесь нет ручных исключений для отдельных запросов. Оставшиеся ошибки не исправлялись подменой item_id.
Для дальнейшей работы интересны более полное представление длинных описаний и отдельная проверка
удалённых услуг, но они не входят в приложенную модель.

## 10. Независимая проверка отправленного CSV

Проверяются ровно две колонки, полный набор query_id в исходном регистре, 50 уникальных кандидатов
на запрос, принадлежность item_id корпусу, формат строк и пробелы. Дополнительно сверяется SHA-256:
он связывает этот файл с результатом 0.882835 на платформе.

In [14]:
from validate_submission import validate

submission = validate(ROOT / 'answer.csv', DATA_DIR / 'benchmark_queries.parquet',
                      DATA_DIR / 'benchmark_items.parquet', require_50=True)
assert submission['valid'], submission['errors']
assert submission['sha256'] == EXPECTED_ANSWER_SHA
display(pd.DataFrame([
    {'проверка': 'Строк и уникальных query_id', 'результат': submission['rows']},
    {'проверка': 'Минимум кандидатов', 'результат': submission['min_candidates']},
    {'проверка': 'Максимум кандидатов', 'результат': submission['max_candidates']},
    {'проверка': 'Всего записанных item_id', 'результат': submission['total_candidates']},
    {'проверка': 'Ошибок формата', 'результат': submission['error_count']},
    {'проверка': 'UTF-8 BOM отсутствует', 'результат': not submission['utf8_bom']},
]))
print('SHA-256 отправленного файла:', submission['sha256'])

,проверка,результат
0,Строк и уникальных query_id,2452
1,Минимум кандидатов,50
2,Максимум кандидатов,50
3,Всего записанных item_id,122600
4,Ошибок формата,0
5,UTF-8 BOM отсутствует,True


SHA-256 отправленного файла: ef4ba4b73e9cbb7821d4b7c2a1899b97f8c962adde06a81a70f81dd51c9910ea


## 11. Рассчитать тот же ответ заново

При `RUN_REPRODUCE = True` следующая ячейка запускает полный pipeline:
построение индексов по исходным таблицам, локальный E5, вычисление признаков и отбор 50 кандидатов.
Ответ записывается в `results/reproduced/answer.csv`. Приложенный CSV не является входным
признаком или источником ответов для расчёта.

Для контрольного запуска используется CPU и версии библиотек из requirements.
Первый расчёт заметно дольше quick-mode; следующие запуски могут переиспользовать кэш.
Точное совпадение не предполагается: `reproduce.py` проверяет его и завершится ошибкой,
если SHA-256 будет отличаться.

Для контрольного запуска используйте пустой кэш. При смене кода, весов или устройства
укажите новый `--cache-dir`: устройство не входит в текущий ключ кэша.

In [15]:
if RUN_REPRODUCE:
    command = [
        sys.executable, 'reproduce.py',
        '--data-dir', str(DATA_DIR.resolve()),
        '--model-dir', 'models',
        '--output-dir', 'results/reproduced',
        '--cache-dir', 'cache/predict',
        '--device', DEVICE,
    ]
    print('Запущен полный расчёт. Это может занять значительное время.')
    # Не сохраняем в notebook подробный stdout с локальными путями.
    completed = subprocess.run(command, cwd=ROOT, text=True, capture_output=True)
    if completed.returncode != 0:
        raise RuntimeError(
            'Повторный расчёт не завершился успешно. Запустите reproduce.py из терминала '
            'для подробного вывода и проверьте версии библиотек, веса и свободную память.')
    reproduced = read_json('results/reproduced/reproduction_report.json')
    assert reproduced['valid'] and reproduced['byte_identical_to_submitted']
    assert reproduced['sha256'] == EXPECTED_ANSWER_SHA
    print('Рассчитанный CSV прошёл форматную проверку и побайтно совпал с отправленным.')
    print('SHA-256:', reproduced['sha256'])
else:
    print('Повторный расчёт не запускался: RUN_REPRODUCE = False.')
    print('Для запуска задайте True в первой кодовой ячейке и выполните Run All.')

Повторный расчёт не запускался: RUN_REPRODUCE = False.
Для запуска задайте True в первой кодовой ячейке и выполните Run All.


## 12. Повторное обучение и файлы решения

Обычное воспроизведение ответа использует готовый бустинг. Для проверки обучения код также приложен.
Чтобы сохранить исходную модель, сначала нужно создать копию каталога моделей:

```bash
cp -R models models_retrained
python train_model.py --data-dir data --model-dir models_retrained --cache-dir cache/train --device cpu
python reproduce.py --data-dir data --model-dir models_retrained --output-dir results/retrained --device cpu
```

Повторить процедуру отложенной проверки можно отдельным `evaluate_model.py`. Она не запускается
по умолчанию в этом ноутбуке и не используется здесь для нового подбора параметров:

```bash
python evaluate_model.py --data-dir data --model-dir models --cache-dir cache/train --output-dir results/holdout_reproduced --device cpu
```

За что отвечают основные файлы:

| Файл | Задача |
|---|---|
| `retrieval.py`, `extra_retrieval.py` | Нормализация, индексы и поисковые источники |
| `intent_model.py`, `rerank_model.py` | Классификатор микрокатегорий и признаки пары |
| `splits.py`, `train_model.py` | Разбиение и воспроизведение обучения |
| `solution_pipeline.py` | Полный расчёт кандидатов и ответа |
| `reproduce.py` | Предсказание, проверка CSV и точного SHA-256 |
| `validate_submission.py` | Проверка ответа без загрузки поисковой модели |
| `setup_assets.py` | Подготовка и проверка локальных файлов E5 |
| `report.md` | Текстовое описание решения |

Используются NumPy, pandas, SciPy, scikit-learn, NLTK, PyArrow, PyTorch,
Sentence Transformers и Transformers. E5 — открытая модель под лицензией MIT;
карточка авторов и зафиксированная версия сохранены в `models/`.

Проверка повторного fit выполнялась на уже рассчитанных признаках: предсказания совпали.
Полная цепочка обучения с нуля с повторным расчётом E5 на CPU отдельно не проверялась;
побайтное совпадение pickle не утверждается. Для воспроизведения отправленного ответа
используется готовый ranker. Подробности сохранены в `results/retraining_check.json`.